# 04 改写 forward：让上半部分的层多跑几遍

**场景**：循环深度（looped / recurrent-depth）模型在每个 token 上把一段层重复跑好几遍，用更多计算
换更强的推理，而不增加参数，比如 Huginn（Geiping et al. 2025）和 Ouro（2025）。Ouro 还让每个 token
自己决定什么时候停。想在真实服务里验证这类结构，你得重写 vLLM 的模型 forward，还得想清楚每一遍
循环的 KV 缓存怎么处理：写进去？覆盖？还是只读不写？

`@pllm.custom_forward` 让你用普通 Python 重写一个 token 的 forward：模型的层被包装成可以按区间调用的
函数，层与层之间写你自己的张量数学；KV 怎么处理用一个参数说清楚。

**先说清楚**：Qwen2.5 没有为循环训练过，把它的层多跑几遍不会让它更聪明。这一篇演示的是**结构**：
同样的代码可以原封不动地用在一个为循环训练过的同家族检查点上。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import gc
from dataclasses import dataclass

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

## 第一步：一个循环负载

`think` 方法收到 `layers`（`pllm.model.Layers`）：

- `layers()` 跑完整个模型，返回每一层之后的残差流 `taps`，`taps[-1]` 是最后一层的输出；
- `layers(start=s, hidden=h)` 从第 s 层开始、以 `h` 为输入跑到最后一层；
- 方法的返回值就是送去最终 norm 和 LM head 的那个向量。

每一次 `layers(...)` 调用都会算出这些层的 K、V。默认 `kv="commit"`：把它们写进 KV 缓存。循环里每一遍都写，后一遍覆盖前一遍，所以后面的 token 看到的是这个 token 最后一遍循环的 K、V。Ouro 论文（arXiv 2510.25741，§5.4.2 表 14）比较了 decode 时循环的几种 KV 用法：每一遍各存一份时 GSM8K 78.92；只留最后一遍 78.85；只留第一遍只有 18.73。每遍都写回就是“只留最后一遍”；如果第一遍写、之后只读不写，相当于“只留第一遍”，会让一个按循环训练过的模型崩掉。`kv="peek"` 读之前位置的缓存加上自己这一遍新算的 K、V，但不写回；它适合“往前试算一步、不留痕迹”的场景（例如投机式的预演），不用于这里的循环。

`loops` 是循环次数，`start` 是被循环的那段层的起点。为了让没训练过循环的模型不至于崩掉，每一遍的结果只按比例 `mix` 混进去（`mix=1` 就是纯粹的循环）。`tol` 是 Ouro 式的逐 token 早停：某个 token 一遍循环带来的相对变化小于 `tol`，之后的循环对它就不再生效。注意循环次数是固定的 Python `for`，早停用 `torch.where` 表达，而不是 `if`：forward 里的控制流不能依赖张量的值，每个 token 的计算量有上界。

In [2]:
@dataclass(frozen=True)
class LatentLoop(pllm.Workload):
    start: int = 20
    loops: int = 0
    mix: float = 1.0
    tol: float = 0.0

    @pllm.custom_forward(tokens="all")
    def think(self, layers: pllm.model.Layers) -> torch.Tensor:
        hidden = layers()[-1]
        done = torch.zeros_like(hidden[:, :1], dtype=torch.bool)
        for _ in range(self.loops):
            again = layers(start=self.start, hidden=hidden)[-1]  # kv="commit"：每一遍都写回 KV
            step = torch.lerp(hidden, again, self.mix)
            change = (step - hidden).float().norm(dim=-1, keepdim=True) / hidden.float().norm(dim=-1, keepdim=True)
            hidden = torch.where(done, hidden, step)
            done = done | (change < self.tol)
        return hidden

## 第二步：`loops=0` 就是原来的模型

不循环时，`think` 只是把整个模型跑一遍。先确认它和原生 vLLM 逐字一致：这说明 `custom_forward` 本身
没有改变任何计算。

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["What is the capital of France? Answer in one sentence.", "Name three primary colors."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=32, temperature=0.0)

plain = pllm.make_llm(CONFIG)
baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]
del plain
gc.collect()
torch.cuda.empty_cache()


def run(workload: LatentLoop) -> list[str]:
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        return [output.outputs[0].text for output in llm.generate(prompts, greedy)]


texts = run(LatentLoop(loops=0))
print("与原生 vLLM 相同:", texts == baseline)
for text in texts:
    print("   ", repr(text))

与原生 vLLM 相同: True
    'The capital of France is Paris.'
    'Three primary colors are red, blue, and yellow.'


## 第三步：循环起来

三组设置：从第 20 层起纯循环一遍（`mix=1`）；从第 20 层起循环一遍、只混入 10%；从第 20 层起循环三遍、每遍混入 10%，并打开早停（`tol=0.05`）（这个 0.5B 模型共 24 层，编号 0–23）。

In [4]:
for workload in (LatentLoop(loops=1, mix=1.0), LatentLoop(loops=1, mix=0.1), LatentLoop(loops=3, mix=0.1, tol=0.05)):
    print(workload)
    for text in run(workload):
        print("   ", repr(text))

LatentLoop(start=20, loops=1, mix=1.0, tol=0.0)


    'The\xa0capital of the\xa0, the\xa0, and the 1,116 other ordinary, ordinary, ordinary, ordinary, and, and, and'
    '«I, 🙂\\, would indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed,\xa0‘‘‘‘‘‘'
LatentLoop(start=20, loops=1, mix=0.1, tol=0.0)


    'The capital of France is Paris. It is the largest city in the country and the administrative center of the country. The city is known for its historical landmarks,'
    'The three primary colors are:\n\n1. Red\n2. Blue\n3. Blue\n\nThese three colors are the most common and commonly used in the field of'
LatentLoop(start=20, loops=3, mix=0.1, tol=0.05)


    'The capital of the country is indeed, and is indeed, the city of the same name, which is the capital of the country, the Republic of the Republic'
    'The three primary colors are: **Red, Blue, and Yellow.»»»»»»»»»»»»»»»»»»'


三组结果：

- **纯循环**（`mix=1`）让没训练过循环的模型崩掉：第 20–23 层从没见过自己的输出作为输入，而且后面的 token 读到的是这一遍写下的 K、V；
- **只混入 10% 的一遍循环**：回答仍通顺，第一个回答比原来长了（多说了几句巴黎），第二个回答把黄色说成了第二个 Blue；
- **循环三遍、每遍 10%**：第一个回答丢了 "France / Paris"，第二个回答末尾出现重复符号；开始偏离。

在这个没为循环训练过的模型上，循环越强，回答偏离越多；要比较循环次数、起点和早停阈值，得换上为循环训练过的检查点，代码不用改。

## 运行时的代价

`custom_forward` 每个 token 都在 Python 描述的程序里重跑层，目前不在 CUDA graph 里运行，pllm 自动用 eager
模式并在 `diagnostics().eager_reason` 里写明原因；它也暂时只支持单卡（`tensor_parallel_size=1`），
以及 KV 全部在分页缓存里的模型（稠密 Qwen2/Llama/Qwen3 和 Qwen3-MoE；Qwen3.5 的 DeltaNet 层有缓存之外的
递归状态，不支持）。

## 还能写什么

- **把上一步的 hidden 当这一步的输入**（Coconut 式连续思维）：声明一个 `pllm.PositionState`，用
  `layers.save(...)` 存下每个位置的最后一层输出，下一个 token 用 `layers.load(...)` 读回来，作为
  `layers(hidden=...)` 的输入。现成例子是 `pllm.workloads.StepLoopWorkload`；
- **层间再循环**：把深层的输出按范数匹配后混回浅层，再跑一遍上面的层，见 `pllm.workloads.RecirculationWorkload`；
- `@pllm.custom_forward(tokens="prefill", over="span")` 把整段 prompt 一起交给你，可以写 Jacobi 式的并行迭代，
  见 `pllm.workloads.JacobiRecirculationWorkload`。

下一篇 [05 反馈循环](05_host_feedback.ipynb) 回到训练：生成、投票、更新在同一个引擎里闭环。